In [ ]:
!nvidia-smi -L
import torch; print(torch.__version__, torch.cuda.is_available())

In [ ]:
# Deep BSDE under multi-asset Heston: PyTorch implementation (GPU), stacked independent runs.
# Port of the validated JavaScript reference implementation (lib.js).
import math, json, time
import torch

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
F32 = torch.float32
F64 = torch.float64
BASE = dict(S0=100.0, K=100.0, r=0.05, T=0.5, kappa=2.0, theta=0.04, rho=-0.7, v0=0.04, rhoS=0.0)


def col(vals, dtype=F32):
    return torch.tensor(vals, dtype=dtype, device=dev).view(-1, 1, 1)


def draw(gen, R, M, d, dt, rho, rhoS, dtype=F32):
    sq = math.sqrt(dt)
    eta = torch.randn(R, M, d, generator=gen, device=dev, dtype=dtype)
    if rhoS > 0:
        eta0 = torch.randn(R, M, 1, generator=gen, device=dev, dtype=dtype)
        w = sq * (math.sqrt(rhoS) * eta0 + math.sqrt(1 - rhoS) * eta)
    else:
        w = sq * eta
    b = rho * w + math.sqrt(1 - rho * rho) * sq * torch.randn(R, M, d, generator=gen, device=dev, dtype=dtype)
    return w, b


def step(scheme, S, v, dW, dB, dt, r, kappa, theta, sigma):
    vp = torch.clamp(v, min=0.0)
    sv = torch.sqrt(vp)
    if scheme == 'PT':
        S_new = S + r * S * dt + sv * S * dW
        v_new = v + kappa * (theta - v) * dt + sigma * sv * dB
    else:  # FT: full truncation + log-Euler
        S_new = S * torch.exp((r - 0.5 * vp) * dt + sv * dW)
        v_new = v + kappa * (theta - vp) * dt + sigma * sv * dB
    return S_new, v_new


def _accumulate(acc, S, d, K, r, T, S0, EB):
    """acc: dict of float64 tensors [R]; S: [R,M,d] terminal prices."""
    S = S.to(F64)
    disc = math.exp(-r * T)
    B = S.mean(dim=2)                     # [R,M]
    g = disc * torch.clamp(B - K, min=0)
    h = B - EB
    dl = torch.where(B > K, disc * S.sum(dim=2) / (d * d * S0), torch.zeros_like(B))
    for k, x in (('g', g), ('gg', g * g), ('h', h), ('hh', h * h), ('gh', g * h), ('dl', dl), ('dl2', dl * dl)):
        acc[k] = acc.get(k, 0) + x.sum(dim=1)
    acc['n'] = acc.get('n', 0) + S.shape[1]


def _finish(acc):
    n = acc['n']
    mg, mh = acc['g'] / n, acc['h'] / n
    vg, vh = acc['gg'] / n - mg ** 2, acc['hh'] / n - mh ** 2
    cgh = acc['gh'] / n - mg * mh
    beta = cgh / vh
    price = mg - beta * mh
    varcv = vg - cgh ** 2 / vh
    md = acc['dl'] / n
    out = []
    for i in range(len(mg)):
        out.append(dict(price=price[i].item(), se=math.sqrt(max(varcv[i].item(), 0) / n), plainPrice=mg[i].item(),
                        plainSe=math.sqrt(max(vg[i].item(), 0) / n), payoffStd=math.sqrt(max(vg[i].item(), 0)),
                        delta=md[i].item(), deltaSe=math.sqrt(max((acc['dl2'][i] / n - md[i] ** 2).item(), 0) / n), nPaths=n))
    return out


def mc_disc(models, scheme, N, nPaths, seed, chunk=None):
    """Stacked MC of discretised dynamics. models: list of dicts sharing d, rhoS."""
    d, rhoS = models[0]['d'], models[0].get('rhoS', 0.0)
    p = {**BASE, **models[0]}
    R = len(models)
    chunk = chunk or max(1024, int(2_000_000 // (R * d)))
    gen = torch.Generator(device=dev); gen.manual_seed(seed)
    dt = p['T'] / N
    sigma = col([m['sigma'] for m in models]); kappa = col([m.get('kappa', BASE['kappa']) for m in models])
    EB = p['S0'] * ((1 + p['r'] * dt) ** N if scheme == 'PT' else math.exp(p['r'] * p['T']))
    acc = {}; done = 0
    while done < nPaths:
        M = min(chunk, nPaths - done)
        S = torch.full((R, M, d), p['S0'], device=dev, dtype=F32)
        v = torch.full((R, M, d), p['v0'], device=dev, dtype=F32)
        for _ in range(N):
            dW, dB = draw(gen, R, M, d, dt, p['rho'], rhoS)
            S, v = step(scheme, S, v, dW, dB, dt, p['r'], kappa, p['theta'], sigma)
        _accumulate(acc, S, d, p['K'], p['r'], p['T'], p['S0'], EB)
        done += M
    return _finish(acc)


def mc_qe(models, N, nPaths, seed, chunk=None):
    """Andersen (2008) QE with martingale correction, gamma1=gamma2=1/2. rhoS must be 0."""
    d = models[0]['d']; p = {**BASE, **models[0]}
    assert not p.get('rhoS'), 'QE needs rhoS=0'
    R = len(models)
    chunk = chunk or max(1024, int(2_000_000 // (R * d)))
    gen = torch.Generator(device=dev); gen.manual_seed(seed)
    dt = p['T'] / N; r, theta, rho = p['r'], p['theta'], p['rho']
    sig = col([m['sigma'] for m in models], F64); kap = col([m.get('kappa', BASE['kappa']) for m in models], F64)
    E = torch.exp(-kap * dt); c1 = sig ** 2 * E * (1 - E) / kap; c2 = theta * sig ** 2 * (1 - E) ** 2 / (2 * kap)
    K1 = 0.5 * dt * (kap * rho / sig - 0.5) - rho / sig; K2 = 0.5 * dt * (kap * rho / sig - 0.5) + rho / sig
    K3 = 0.5 * dt * (1 - rho * rho); K4 = K3; A = K2 + 0.5 * K4
    E, c1, c2, K1, K2, A = [x.to(F32) for x in (E, c1, c2, K1, K2, A)]
    EB = p['S0'] * math.exp(r * p['T'])
    acc = {}; done = 0
    while done < nPaths:
        M = min(chunk, nPaths - done)
        lnS = torch.full((R, M, d), math.log(p['S0']), device=dev, dtype=F32)
        v = torch.full((R, M, d), p['v0'], device=dev, dtype=F32)
        for _ in range(N):
            m = theta + (v - theta) * E
            s2 = v * c1 + c2
            psi = s2 / (m * m)
            Z = torch.randn(R, M, d, generator=gen, device=dev, dtype=F32)
            U = torch.rand(R, M, d, generator=gen, device=dev, dtype=F32)
            Zs = torch.randn(R, M, d, generator=gen, device=dev, dtype=F32)
            ip = 2.0 / psi
            b2 = ip - 1 + torch.sqrt(ip) * torch.sqrt(torch.clamp(ip - 1, min=0))
            a = m / (1 + b2)
            vq = a * (torch.sqrt(torch.clamp(b2, min=0)) + Z) ** 2
            den = torch.clamp(1 - 2 * A * a, min=1e-12)
            Mq = torch.exp(A * b2 * a / den) / torch.sqrt(den)
            pp = (psi - 1) / (psi + 1)
            beta = (1 - pp) / m
            ve = torch.where(U <= pp, torch.zeros_like(U), torch.log(torch.clamp((1 - pp) / (1 - U), min=1e-30)) / beta)
            Me = pp + beta * (1 - pp) / (beta - A)
            quad = psi <= 1.5
            vn = torch.where(quad, vq, ve)
            Mg = torch.where(quad, Mq, Me)
            K0s = -torch.log(Mg) - (K1 + 0.5 * K3) * v
            lnS = lnS + r * dt + K0s + K1 * v + K2 * vn + torch.sqrt(torch.clamp(K3 * v + K4 * vn, min=0)) * Zs
            v = vn
        _accumulate(acc, torch.exp(lnS), d, p['K'], r, p['T'], p['S0'], EB)
        done += M
    return _finish(acc)


def train_stack(models, seeds, cfg):
    """Train R independent Deep BSDE solvers in parallel (one per (model, seed)).
    All models share d, rhoS; cfg shared: N, batch, iters, width, depth, scheme, warmStart, lr..."""
    R = len(models); d = models[0]['d']; rhoS = models[0].get('rhoS', 0.0); p = {**BASE, **models[0]}
    D2 = 2 * d
    N = cfg.get('N', 40); M = cfg.get('batch', 256); iters = cfg.get('iters', 4000); depth = cfg.get('depth', 2)
    h = cfg.get('width') or (d + 10); scheme = cfg.get('scheme', 'PT')
    dt = p['T'] / N; S0, K, r, theta, v0, rho = p['S0'], p['K'], p['r'], p['theta'], p['v0'], p['rho']
    gen = torch.Generator(device=dev); gen.manual_seed(int(1000003 * seeds[0] + 7919 * R + 13 * d + N))
    sigma = col([m['sigma'] for m in models]); kappa = col([m.get('kappa', BASE['kappa']) for m in models])
    zs = S0 * math.sqrt(theta); Df = 1 + r * dt; sTh = math.sqrt(theta)
    # params
    dims = [D2] + [h] * depth + [D2]
    Ws, bs = [], []
    for l in range(depth + 1):
        sd = math.sqrt(2.0 / dims[l]) * (0.1 if l == depth else 1.0)
        Ws.append((torch.randn(R, N - 1, dims[l], dims[l + 1], generator=gen, device=dev) * sd).requires_grad_())
        bs.append(torch.zeros(R, N - 1, 1, dims[l + 1], device=dev, requires_grad=True))
    if cfg.get('warmStart'):
        y0init = [mc_disc([m], scheme, N, 5000, 999 + s)[0]['plainPrice'] for m, s in zip(models, seeds)]
        Y0 = torch.tensor(y0init, device=dev, dtype=F32)
    else:
        fwd = max(S0 * math.exp(r * p['T']) - K, 1.0)
        Y0 = (0.5 + torch.rand(R, generator=gen, device=dev)) * fwd * 2
    Y0 = Y0.clone().requires_grad_()
    Z0 = torch.zeros(R, D2, device=dev, requires_grad=True)
    base_lr = cfg.get('lr', 3e-3)
    opt = torch.optim.Adam([{'params': [Y0], 'lr': cfg.get('y0lr', 0.05)}, {'params': [Z0], 'lr': cfg.get('z0lr', 0.01)},
                            {'params': Ws + bs, 'lr': base_lr}], betas=(0.9, 0.999), eps=1e-8)
    init_lrs = [g['lr'] for g in opt.param_groups]
    sched = cfg.get('schedule', [(0.5, 1.0), (0.8, 0.1), (1.0, 0.01)])
    powN = torch.tensor([Df ** (N - 1 - n) for n in range(N)], device=dev, dtype=F32).view(1, N, 1)

    def simulate(Mb):
        S = torch.full((R, Mb, d), S0, device=dev); v = torch.full((R, Mb, d), v0, device=dev)
        Xs, dWs = [], []
        for n in range(N):
            Xs.append(torch.cat([(S / S0 - 1) / sTh, (v - theta) / theta], dim=2))
            dW, dB = draw(gen, R, Mb, d, dt, rho, rhoS)
            dWs.append(torch.cat([dW, dB], dim=2))
            S, v = step(scheme, S, v, dW, dB, dt, r, kappa, theta, sigma)
        g = torch.clamp(S.mean(dim=2) - K, min=0)  # [R,Mb]
        return torch.stack(Xs, 1), torch.stack(dWs, 1), g   # [R,N,Mb,2d]

    def rollout(X, dWb, Mb):
        H = X[:, 1:].reshape(R * (N - 1), Mb, D2)
        for l in range(depth + 1):
            H = torch.bmm(H, Ws[l].reshape(R * (N - 1), dims[l], dims[l + 1])) + bs[l].reshape(R * (N - 1), 1, dims[l + 1])
            if l < depth:
                H = torch.relu(H)
        Z = zs * H.reshape(R, N - 1, Mb, D2)
        incr = torch.cat([(zs * Z0).view(R, 1, 1, D2).expand(R, 1, Mb, D2) * dWb[:, :1], Z * dWb[:, 1:]], dim=1).sum(-1)  # [R,N,Mb]
        YN = Y0.view(R, 1) * Df ** N + (incr * powN).sum(1)
        return YN

    t0 = time.time(); hist = []
    for it in range(iters):
        frac = it / iters
        f = next(v for b, v in sched if frac < b)
        for g_, lr0 in zip(opt.param_groups, init_lrs):
            g_['lr'] = lr0 * f
        X, dWb, g = simulate(M)
        YN = rollout(X, dWb, M)
        lossR = ((YN - g) ** 2).mean(1)
        opt.zero_grad(set_to_none=True)
        lossR.sum().backward()
        opt.step()
        if it % 100 == 0 or it == iters - 1:
            hist.append([it, Y0.detach().cpu().tolist(), lossR.detach().cpu().tolist()])
    if dev == 'cuda':
        torch.cuda.synchronize()
    train_s = time.time() - t0
    # validation
    nVal = cfg.get('nVal', 8192); sse = torch.zeros(R, device=dev, dtype=F64); sg = torch.zeros_like(sse); sgg = torch.zeros_like(sse); cnt = 0
    with torch.no_grad():
        while cnt < nVal:
            Mb = 1024
            X, dWb, g = simulate(Mb); YN = rollout(X, dWb, Mb)
            sse += ((YN - g) ** 2).to(F64).sum(1); sg += g.to(F64).sum(1); sgg += (g.to(F64) ** 2).sum(1); cnt += Mb
    mse = (sse / cnt); gm = sg / cnt; gvar = sgg / cnt - gm ** 2
    out = []
    z0 = (zs * Z0).detach().cpu()
    for i in range(R):
        dvec = (z0[i, :d] / (math.sqrt(v0) * S0)).tolist()
        out.append(dict(price=Y0[i].item(), delta=sum(dvec) / d, z0=z0[i].tolist(), valMSE=mse[i].item(), payoffVar=gvar[i].item(),
                        hedgeVarReduction=1 - mse[i].item() / gvar[i].item(),
                        finalTrainLoss=sum(hh[2][i] for hh in hist[-5:]) / 5,
                        trainSecStack=train_s, R=R,
                        nParams=1 + D2 + (N - 1) * sum(dims[l] * dims[l + 1] + dims[l + 1] for l in range(depth + 1)),
                        hist=[[hh[0], round(hh[1][i], 5), round(hh[2][i], 4)] for hh in hist[::5]]))
    return out


def M(d, sigma, kappa=2.0, rhoS=0.0):
    return dict(d=d, sigma=round(sigma, 4), kappa=kappa, rhoS=rhoS)


In [ ]:
import time
t=time.time()
print('QE d=1 (analytic 6.762192, 6.300634; delta .66962 .73271):', [(round(x['price'],4),round(x['se'],4),round(x['delta'],4)) for x in mc_qe([M(1,0.4),M(1,0.8)],50,4_000_000,1)], f'{time.time()-t:.1f}s')
t=time.time()
print('PT40 d=1 sig .4/.8 (JS: 6.7635, 6.3438 +-.003):', [(round(x['price'],4),round(x['se'],4)) for x in mc_disc([M(1,0.4),M(1,0.8)],'PT',40,4_000_000,2)], f'{time.time()-t:.1f}s')
t=time.time()
print('QE d=20 sig .4 (JS: 2.8361+-.0007):', [(round(x['price'],4),round(x['se'],4),round(x['delta'],5)) for x in mc_qe([M(20,0.4)],50,1_000_000,3)], f'{time.time()-t:.1f}s')
t=time.time()
out=train_stack([M(1,0.4),M(1,0.8),M(20,0.1)][:2],[1,1],{'iters':4000})
print('BSDE d=1 (JS: 6.7648/.6709, 6.3471/.7314):', [(round(o['price'],4),round(o['delta'],4),round(o['hedgeVarReduction'],4)) for o in out], f'{time.time()-t:.1f}s')
t=time.time()
out=train_stack([M(20,0.1)]*3,[1,2,3],{'iters':4000})
print('BSDE d=20 sig .1 x3 (true 2.8498; JS 2.8505/2.8534/2.8508, delta .0396):', [(round(o['price'],4),round(o['delta'],5),round(o['hedgeVarReduction'],4)) for o in out], f'{time.time()-t:.1f}s')

In [ ]:
# Experiment driver (run after dbsde_torch definitions are loaded)
import json, time, math

OUT = open('results_gpu.jsonl', 'a')
def emit(kind, payload):
    line = json.dumps({'kind': kind, **payload})
    OUT.write(line + '\n'); OUT.flush()
    print('RES ' + line, flush=True)

def M(d, sigma, kappa=2.0, rhoS=0.0):
    return dict(d=d, sigma=round(sigma, 4), kappa=kappa, rhoS=rhoS)

def paths_for(d):
    return 8_000_000 if d <= 2 else (4_000_000 if d <= 5 else 2_000_000)

def ref(kind, models, **kw):
    t = time.time()
    if kind == 'qe':
        res = mc_qe(models, kw.get('N', 50), kw.get('nPaths', paths_for(models[0]['d'])), kw.get('seed', 101))
    else:
        res = mc_disc(models, kw['scheme'], kw['N'], kw.get('nPaths', paths_for(models[0]['d'])), kw.get('seed', 102 + kw['N']))
    dt = time.time() - t
    for m, r in zip(models, res):
        emit('ref', dict(ref=kind if kind == 'qe' else f"disc:{kw['scheme']}:{kw['N']}", model=m, result=r, stackSec=dt, R=len(models)))

def bsde(tag, models, seeds, cfg):
    t = time.time()
    res = train_stack(models, seeds, cfg)
    for m, s, r in zip(models, seeds, res):
        emit('bsde', dict(tag=tag, model=m, seed=s, cfg=cfg, result=r))
    print(f'# stack {tag} d={models[0]["d"]} R={len(models)} cfg={cfg} took {time.time()-t:.1f}s', flush=True)

SIGS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
T0 = time.time()
# ---------------- references ----------------
for d in [1, 2, 5, 10, 20]:
    ms = [M(d, s) for s in SIGS]
    if d > 1: ref('qe', ms)
    ref('disc', ms, scheme='PT', N=40)
print(f'# E1 refs done {time.time()-T0:.0f}s', flush=True)
# E2 maps: d=1 kappa x sigma fine grid; d=5 coarse
for kappa in [1.0, 2.0, 4.0]:
    ms1 = [M(1, 0.05 * k, kappa) for k in range(2, 21)]
    ref('qe', ms1, nPaths=4_000_000); ref('disc', ms1, scheme='PT', N=40); ref('disc', ms1, scheme='FT', N=40)
    ms5 = [M(5, 0.1 * k, kappa) for k in range(1, 11)]
    ref('qe', ms5); ref('disc', ms5, scheme='PT', N=40); ref('disc', ms5, scheme='FT', N=40)
print(f'# E2 refs done {time.time()-T0:.0f}s', flush=True)
# E3: N and scheme
for d in [1, 5]:
    ms = [M(d, s) for s in [0.3, 0.5, 0.7]]
    for sc in ['PT', 'FT']:
        for N in [10, 20, 40, 80, 160]:
            ref('disc', ms, scheme=sc, N=N)
# E4 extra refs (PT80 at d=2,10)
for d in [2, 10]:
    ref('disc', [M(d, 0.3), M(d, 0.7)], scheme='PT', N=80)
# E5: correlated assets: fine FT400 reference + PT40
for d in [1, 2, 5, 10, 20]:
    ms = [M(d, 0.3, rhoS=0.5), M(d, 0.7, rhoS=0.5)] if d > 1 else [M(1, 0.3), M(1, 0.7)]
    ref('disc', ms, scheme='FT', N=400, nPaths=2_000_000, seed=104)
    if d > 1: ref('disc', ms, scheme='PT', N=40)
print(f'# all refs done {time.time()-T0:.0f}s', flush=True)

# ---------------- Deep BSDE ----------------
# E1 main grid
for d in [1, 2, 5, 10, 20]:
    nS = 5 if d <= 5 else 3
    ms, ss = [], []
    for s in SIGS:
        for seed in range(1, nS + 1):
            ms.append(M(d, s)); ss.append(seed)
    bsde('E1', ms, ss, {})
# E2 Feller shift
ms, ss = [], []
for kappa in [1.0, 4.0]:
    for s in [0.2, 0.4, 0.6, 0.8, 1.0]:
        for seed in [1, 2, 3]:
            ms.append(M(1, s, kappa)); ss.append(seed)
bsde('E2', ms, ss, {})
# E3 N and scheme
for d in [1, 5]:
    combos = [('PT', 10), ('PT', 20), ('PT', 80), ('FT', 10), ('FT', 20), ('FT', 40), ('FT', 80)] if d == 1 else [('PT', 80), ('FT', 40), ('FT', 80)]
    for sc, N in combos:
        ms, ss = [], []
        for s in [0.3, 0.5, 0.7]:
            for seed in [1, 2, 3]:
                ms.append(M(d, s)); ss.append(seed)
        bsde('E3', ms, ss, {'scheme': sc, 'N': N})
# E4 ablations
for d in [2, 10]:
    variants = [{'variant': 'wider', 'width': 4 * (d + 10)}, {'variant': 'deeper', 'depth': 3}, {'variant': 'moreIters', 'iters': 10000},
                {'variant': 'largerBatch', 'batch': 1024}, {'variant': 'moreSteps', 'N': 80}, {'variant': 'warmStart', 'warmStart': True},
                {'variant': 'FTscheme', 'scheme': 'FT'}]
    for v in variants:
        ms, ss = [], []
        for s in [0.3, 0.7]:
            for seed in [1, 2, 3]:
                ms.append(M(d, s)); ss.append(seed)
        bsde('E4', ms, ss, v)
# E5 correlated assets
for d in [2, 5, 10, 20]:
    ms, ss = [], []
    for s in [0.3, 0.7]:
        for seed in [1, 2, 3]:
            ms.append(M(d, s, rhoS=0.5)); ss.append(seed)
    bsde('E5', ms, ss, {})
# Timing: single runs
for d in [1, 2, 5, 10, 20]:
    bsde('TIME', [M(d, 0.4)], [1], {})
print(f'# ALL DONE {time.time()-T0:.0f}s', flush=True)
OUT.close()


In [ ]:
import json
rows=[json.loads(l) for l in open("results_gpu.jsonl")]
print(len(rows), sum(r["kind"]=="bsde" for r in rows))